In [1]:
# ============================================================
# FE5108 Final Project - Part 1
# Question 1A: Factor Premiums and Factor Decay
# ============================================================

import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

# Locate the project root
current_dir = Path.cwd().resolve()
ROOT = next(
    (p for p in [current_dir, *current_dir.parents]
     if (p / "src" / "config.py").exists()),
    None
)

if ROOT is None:
    raise FileNotFoundError("Project root not found.")

sys.path.insert(0, str(ROOT))

from src import config, utils

config.apply_style()

print("Project root:", ROOT)

Project root: /Users/zhangjiaxin/Desktop/fe5018-final


In [2]:
# Step 2: Load the Fama-French three-factor data

raw = pd.read_excel(
    config.FILE_FF_MSCI,
    sheet_name=config.SHEET_FF3,
    header=None
)

display(raw.head(12))

,0,1,2,3,4
0,This file was created by CMPT_ME_BEME_RETS usi...,NaN,NaN,NaN,NaN
1,The 1-month TBill return is from Ibbotson and ...,Inc.,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,Mkt-RF,SMB,HML,RF
4,192607,2.96,-2.56,-2.43,0.22
5,192608,2.64,-1.17,3.82,0.25
6,192609,0.36,-1.4,0.13,0.23
7,192610,-3.24,-0.09,0.7,0.32
8,192611,2.53,-0.1,-0.51,0.31
9,192612,2.62,-0.03,-0.05,0.28


In [3]:
# Step 3: Clean and preprocess monthly factor returns

df = raw.iloc[4:].copy()
df.columns = ["Date", "Mkt-RF", "SMB", "HML", "RF"]

# Retain monthly observations in YYYYMM format
dates = df["Date"].astype(str).str.strip()
mask = dates.str.fullmatch(r"\d{6}(\.0)?")
df = df.loc[mask].copy()

# Convert dates into a monthly PeriodIndex
df["Date"] = df["Date"].astype(float).astype(int).astype(str)
df = df.set_index("Date")

# Convert percentage returns to decimal returns
df = df.apply(pd.to_numeric, errors="coerce")
df = utils.pct_to_decimal(df)
df = utils.to_monthly_index(df)

display(df.head())

print("Sample period:", df.index.min(), "to", df.index.max())
print("Total observations:", len(df))

,Mkt-RF,SMB,HML,RF
month,,,,
1926-07,0.0296,-0.0256,-0.0243,0.0022
1926-08,0.0264,-0.0117,0.0382,0.0025
1926-09,0.0036,-0.0140,0.0013,0.0023
1926-10,-0.0324,-0.0009,0.0070,0.0032
1926-11,0.0253,-0.0010,-0.0051,0.0031


Sample period: 1926-07 to 2023-07
Total observations: 1165


In [4]:
# Step 4: Calculate factor premiums and t-statistics

factors = ["Mkt-RF", "SMB", "HML"]
periods = ["1A_pre", "1A_post"]

results = []

for period in periods:
    sample = utils.select_period(df, period)

    for factor in factors:
        r = sample[factor].dropna()
        t_stat = utils.t_stat_mean(r)

        results.append({
            "Period": period,
            "Factor": factor,
            "N": len(r),
            "Mean Monthly Return (%)": r.mean() * 100,
            "t-stat": t_stat,
            "Significant at 5%": abs(t_stat) > 1.96
        })

result_df = pd.DataFrame(results)

display(result_df.round(4))

,Period,Factor,N,Mean Monthly Return (%),t-stat,Significant at 5%
0,1A_pre,Mkt-RF,792,0.6552,3.2185,True
1,1A_pre,SMB,792,0.2399,2.1258,True
2,1A_pre,HML,792,0.4353,3.3331,True
3,1A_post,Mkt-RF,367,0.7185,3.0914,True
4,1A_post,SMB,367,0.0995,0.5994,False
5,1A_post,HML,367,0.1457,0.8304,False


In [5]:
# Step 5: Validate sample periods and data quality

print("Duplicate months:", df.index.duplicated().sum())

print("\nMissing factor observations:")
display(df[factors].isna().sum())

for period in periods:
    sample = utils.select_period(df, period)

    print(
        f"{period}: {sample.index.min()} to "
        f"{sample.index.max()}, N = {len(sample)}"
    )

assert len(utils.select_period(df, "1A_pre")) == 792
assert len(utils.select_period(df, "1A_post")) == 367
assert df.index.duplicated().sum() == 0
assert df[factors].notna().all().all()

print("\nAll validation checks passed.")

Duplicate months: 0

Missing factor observations:


Mkt-RF    0
SMB       0
HML       0
dtype: int64

1A_pre: 1927-01 to 1992-12, N = 792
1A_post: 1993-01 to 2023-07, N = 367

All validation checks passed.


In [6]:
# Step 6: Export the factor premium comparison table

saved_path = utils.save_table(
    result_df.set_index(["Period", "Factor"]),
    "tab_1A_factor_premiums"
)

print("Results saved to:", saved_path)

Results saved to: /Users/zhangjiaxin/Desktop/fe5018-final/output/tables/tab_1A_factor_premiums.csv


In [7]:
# Step 7: Inspect the momentum and beta portfolio data

sheets = {
    "Momentum": config.SHEET_FF_MOM,
    "Beta": config.SHEET_FF_BETA
}

for factor_name, sheet_name in sheets.items():
    raw_factor = pd.read_excel(
        config.FILE_FF_MSCI,
        sheet_name=sheet_name,
        header=None
    )

    print(f"\n{'=' * 55}")
    print(f"{factor_name.upper()} PORTFOLIOS")
    print(f"Sheet: {sheet_name}")
    print(f"Shape: {raw_factor.shape}")
    print(f"{'=' * 55}")

    display(raw_factor.head(18))


MOMENTUM PORTFOLIOS
Sheet: FF - portfolios sorted on pmom
Shape: (4959, 11)


,0,1,2,3,4,5,6,7,8,9,10
0,This file was created by CMPT_PRIOR_RETS using...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,It contains value- and equal-weighted returns ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,The portfolios are constructed monthly. PRIOR_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,The annual returns are from January to December.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Missing data are indicated by -99.99 or -999.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Average Value Weighted Returns -- Monthly,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



BETA PORTFOLIOS
Sheet: FF - portfolios sorted on beta
Shape: (3105, 16)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,This file was created by CMPT_BETA_RETS using ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,It contains value- and equal-weighted returns ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,The portfolios are constructed at the end of J...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,(requiring at least 24 months with non-missin...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Annual returns are from January to December.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Missing data are indicated by -99.99 or -999.,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,The break points include utilities and include...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Step 8: Extract value-weighted monthly portfolio returns

def extract_vw_monthly(sheet_name, header_row, low_col, high_col):
    raw = pd.read_excel(
        config.FILE_FF_MSCI,
        sheet_name=sheet_name,
        header=None
    )

    # Locate the next data section to avoid mixing return frequencies
    section_labels = raw.iloc[:, 0].astype(str)
    next_sections = section_labels[
        (section_labels.index > header_row) &
        section_labels.str.contains(
            r"Weighted Returns|Annual|Number of Firms",
            case=False,
            na=False
        )
    ]
    end_row = next_sections.index.min() if len(next_sections) else len(raw)

    # Select the monthly date and extreme decile columns
    data = raw.iloc[
        header_row + 1:end_row,
        [0, low_col, high_col]
    ].copy()

    data.columns = ["Date", "Low", "High"]

    # Retain observations with a six-digit monthly date
    date_text = data["Date"].astype(str).str.strip()
    mask = date_text.str.fullmatch(r"\d{6}(\.0)?")
    data = data.loc[mask].copy()

    data["Date"] = (
        data["Date"]
        .astype(float)
        .astype(int)
        .astype(str)
    )

    data = data.set_index("Date")

    # Treat the Fama-French missing-value codes as missing
    data = data.apply(pd.to_numeric, errors="coerce")
    data = data.replace([-99.99, -999], float("nan"))

    # Convert percent returns to monthly decimal returns
    data = utils.pct_to_decimal(data)
    data = utils.to_monthly_index(data)

    return data


momentum = extract_vw_monthly(
    sheet_name=config.SHEET_FF_MOM,
    header_row=10,
    low_col=1,
    high_col=10
)

beta = extract_vw_monthly(
    sheet_name=config.SHEET_FF_BETA,
    header_row=15,
    low_col=6,
    high_col=15
)

print("Momentum portfolios:")
display(momentum.head())

print("Beta portfolios:")
display(beta.head())

Momentum portfolios:


,Low,High
month,,
1927-01,-0.0332,-0.0024
1927-02,0.0753,0.0701
1927-03,-0.0323,0.0547
1927-04,0.0204,0.0549
1927-05,0.0272,0.0632


Beta portfolios:


,Low,High
month,,
1963-07,0.0136,-0.0077
1963-08,0.0352,0.1047
1963-09,-0.0310,-0.0368
1963-10,0.0124,0.0301
1963-11,-0.0091,0.0052


In [10]:
# Step 9: Calculate monthly factor return spreads

momentum_spread = momentum["High"] - momentum["Low"]
beta_spread = beta["Low"] - beta["High"]

momentum_spread.name = "Momentum"
beta_spread.name = "LowMinusHighBeta"

# Align both factors to the period required for later analysis
factor_spreads = pd.concat(
    [momentum_spread, beta_spread],
    axis=1
).loc["1995-01":"2023-07"]

print("Factor spread sample:")
display(factor_spreads.head(10))

print("\nSample period:")
print(factor_spreads.index.min(), "to", factor_spreads.index.max())

print("\nNumber of observations:")
print(factor_spreads.count())

print("\nMissing values:")
print(factor_spreads.isna().sum())

Factor spread sample:


,Momentum,LowMinusHighBeta
month,,
1995-01,-0.0346,0.0122
1995-02,0.0114,-0.0272
1995-03,0.0360,0.0063
1995-04,0.0194,0.0309
1995-05,-0.0011,-0.0059
1995-06,0.0011,-0.0719
1995-07,0.0076,-0.0513
1995-08,-0.0041,-0.0359
1995-09,0.0497,-0.0075



Sample period:
1995-01 to 2023-07

Number of observations:
Momentum            343
LowMinusHighBeta    343
dtype: int64

Missing values:
Momentum            0
LowMinusHighBeta    0
dtype: int64


In [12]:
# Verify the portfolio spread calculations against raw data

assert abs(momentum_spread.loc["1927-01"] - 0.0308) < 1e-10
assert abs(beta_spread.loc["1963-07"] - 0.0213) < 1e-10

print("Momentum and Low-beta spread checks passed.")

Momentum and Low-beta spread checks passed.


In [11]:
# Step 10: Validate and export monthly factor spreads

assert factor_spreads.index.min() == pd.Period("1995-01", freq="M")
assert factor_spreads.index.max() == pd.Period("2023-07", freq="M")
assert len(factor_spreads) == 343
assert factor_spreads.index.is_unique
assert factor_spreads.notna().all().all()

saved_path = utils.save_table(
    factor_spreads,
    "tab_1A_factor_spreads"
)

print("All validation checks passed.")
print("Factor spreads saved to:", saved_path)

All validation checks passed.
Factor spreads saved to: /Users/zhangjiaxin/Desktop/fe5018-final/output/tables/tab_1A_factor_spreads.csv


In [13]:
# Step 11: Prepare the final factor premium comparison exhibit

exhibit_1A = result_df.copy()

exhibit_1A["Period"] = exhibit_1A["Period"].replace({
    "1A_pre": "1927–1992",
    "1A_post": "1993–2023"
})

exhibit_1A["Mean Monthly Return (%)"] = (
    exhibit_1A["Mean Monthly Return (%)"].round(3)
)

exhibit_1A["t-stat"] = exhibit_1A["t-stat"].round(3)

exhibit_1A["Significance"] = exhibit_1A[
    "Significant at 5%"
].map({
    True: "Significant",
    False: "Not significant"
})

exhibit_1A = exhibit_1A[
    ["Factor", "Period", "N",
     "Mean Monthly Return (%)", "t-stat", "Significance"]
]

display(exhibit_1A)

,Factor,Period,N,Mean Monthly Return (%),t-stat,Significance
0,Mkt-RF,1927–1992,792,0.655,3.219,Significant
1,SMB,1927–1992,792,0.240,2.126,Significant
2,HML,1927–1992,792,0.435,3.333,Significant
3,Mkt-RF,1993–2023,367,0.719,3.091,Significant
4,SMB,1993–2023,367,0.099,0.599,Not significant
5,HML,1993–2023,367,0.146,0.830,Not significant


In [14]:
# Verify that both Question 1A output files exist

files = [
    config.TAB_DIR / "tab_1A_factor_premiums.csv",
    config.TAB_DIR / "tab_1A_factor_spreads.csv"
]

for file in files:
    print(f"{file.name}: {'Found' if file.exists() else 'Missing'}")

tab_1A_factor_premiums.csv: Found
tab_1A_factor_spreads.csv: Found
